In [ ]:
from pyspark.sql import functions as F

account = "bikestores@myolympicdatastorage.dfs.core.windows.net"
silver = f"abfss://{account}/silver"
gold = f"abfss://{account}/gold"

log_rows = []

def log(table, step, rows_in, rows_rejected, rows_out):
    log_rows.append((table, step, rows_in, rows_rejected, rows_out))
    print(f"[{table}] {step}: read={rows_in}, rejected={rows_rejected}, written={rows_out}")

In [ ]:
orders      = spark.read.format("delta").load(f"{silver}/orders")
items       = spark.read.format("delta").load(f"{silver}/order_items")
products    = spark.read.format("delta").load(f"{silver}/products")
customers   = spark.read.format("delta").load(f"{silver}/customers")
stores      = spark.read.format("delta").load(f"{silver}/stores")
brands      = spark.read.format("delta").load(f"{silver}/brands")
categories  = spark.read.format("delta").load(f"{silver}/categories")

n_items = items.count()

sales = (items
    .join(orders.select("order_id", "customer_id", "store_id", "order_date"), "order_id", "inner")
    .join(products.select("product_id", "product_name", "brand_id", "category_id"), "product_id", "inner")
    .join(brands, "brand_id", "left")
    .join(categories, "category_id", "left")
    .join(customers.select("customer_id", "first_name", "last_name", "state"), "customer_id", "left")
    .join(stores.select("store_id", F.col("store_name"), F.col("state").alias("store_state")), "store_id", "left")
    .withColumn("revenue", F.col("quantity") * F.col("list_price") * (1 - F.col("discount")))
)

log("sales_fact", "join", n_items, n_items - sales.count(), sales.count())
sales.cache()

In [ ]:
daily = (sales.groupBy("order_date")
         .agg(F.round(F.sum("revenue"), 2).alias("total_revenue"),
              F.countDistinct("order_id").alias("total_orders"),
              F.sum("quantity").alias("units_sold")))
log("daily_sales", "aggregate", sales.count(), 0, daily.count())
daily.write.mode("overwrite").format("delta").save(f"{gold}/daily_sales")

monthly = (sales.withColumn("year", F.year("order_date"))
           .withColumn("month", F.month("order_date"))
           .groupBy("year", "month")
           .agg(F.round(F.sum("revenue"), 2).alias("total_revenue"),
                F.countDistinct("order_id").alias("total_orders"),
                F.sum("quantity").alias("units_sold")))
log("monthly_sales", "aggregate", sales.count(), 0, monthly.count())
monthly.write.mode("overwrite").format("delta").save(f"{gold}/monthly_sales")

In [ ]:
top_products = (sales.groupBy("product_id", "product_name", "brand_name", "category_name")
                .agg(F.round(F.sum("revenue"), 2).alias("total_revenue"),
                     F.sum("quantity").alias("units_sold"))
                .orderBy(F.desc("total_revenue")))
log("top_products", "aggregate", sales.count(), 0, top_products.count())
top_products.write.mode("overwrite").format("delta").save(f"{gold}/top_products")

In [ ]:
clv = (sales.groupBy("customer_id", "first_name", "last_name", "state")
       .agg(F.round(F.sum("revenue"), 2).alias("lifetime_value"),
            F.countDistinct("order_id").alias("total_orders"),
            F.min("order_date").alias("first_order"),
            F.max("order_date").alias("last_order")))
log("customer_lifetime_value", "aggregate", sales.count(), 0, clv.count())
clv.write.mode("overwrite").format("delta").save(f"{gold}/customer_lifetime_value")

In [ ]:
store_sales = (sales.groupBy("store_id", "store_name", "store_state")
               .agg(F.round(F.sum("revenue"), 2).alias("total_revenue"),
                    F.countDistinct("order_id").alias("total_orders")))
log("store_sales", "aggregate", sales.count(), 0, store_sales.count())
store_sales.write.mode("overwrite").format("delta").save(f"{gold}/store_sales")

In [ ]:
display(spark.createDataFrame(log_rows, ["table", "step", "rows_read", "rows_rejected", "rows_written"]))
display(dbutils.fs.ls(gold))